# 23 — Skull Fracture 2.5D Expanded-Negative Series Classifier

## Goal
Train a fast fracture model aligned with the actual project target: **series-level probability of any skull fracture**.

Strategy:
- 2.5D bone-window input: previous / center / next slice.
- Positive-series slices use provided box annotations.
- Every slice from a truly fracture-negative series is a valid negative.
- Unannotated slices from fracture-positive series are not used as negative targets.
- Series probability is calibrated on DEV using multiple aggregators (`max`, top-k mean).

This is intended to complement or replace the heavier detector if it gives better series-level F1.

## 1. Environment

In [ ]:
!pip install -q pydicom pylibjpeg pylibjpeg-libjpeg

## 2. Imports

In [ ]:
import json, random, time, warnings
from functools import lru_cache
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from pydicom.pixels import apply_modality_lut
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", message="Invalid value for VR UI.*")

## 3. Configuration

In [ ]:
SEED = 20260918
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TEST, N_DEV, N_SPLIT_TRIALS = 68, 54, 1500
IMAGE_SIZE = 384
BONE_CENTER, BONE_WIDTH = 600.0, 2800.0
BATCH_SIZE, NUM_WORKERS = 12, 2
EPOCHS, PATIENCE = 20, 6
SAMPLES_PER_EPOCH = 7000
LR, WEIGHT_DECAY = 2e-4, 1e-4
CONTEXT_OFFSETS = (-1, 0, 1)
HU_CACHE_SIZE = 128

DATASET_ROOT_CANDIDATES = [Path("/kaggle/input/datasets/mehdipaykanheyrati/iaaa-contest-bct"), Path("/kaggle/input/iaaa-contest-bct")]
OUTPUT_ROOT = Path("/kaggle/working/fracture_2p5d_expanded_negatives")
MODELS_DIR = OUTPUT_ROOT / "models"; METRICS_DIR = OUTPUT_ROOT / "metrics"; CACHE_DIR = OUTPUT_ROOT / "cache"
for p in [MODELS_DIR, METRICS_DIR, CACHE_DIR]: p.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)

## 4. Locate data and reconstruct the common split

In [ ]:
def first_existing(paths): return next((p for p in paths if p.exists()), None)
def norm_id(v):
    try: return str(int(float(v)))
    except Exception: return str(v).strip()

DATASET_ROOT = first_existing(DATASET_ROOT_CANDIDATES)
if DATASET_ROOT is None: raise FileNotFoundError("CT dataset not found.")
DATA_ROOT = first_existing([DATASET_ROOT/"iaaa-contest-bct"/"Data", DATASET_ROOT/"Data"])
TRAINING_DIR, ANNOTATIONS_DIR = DATA_ROOT/"training", DATA_ROOT/"annotations"
TARGETS_PATH = first_existing([DATASET_ROOT/"series_targets_df.csv", DATASET_ROOT/"iaaa-contest-bct"/"series_targets_df.csv", DATA_ROOT/"series_targets_df.csv", DATA_ROOT.parent/"series_targets_df.csv"])
if TARGETS_PATH is None: raise FileNotFoundError("series_targets_df.csv not found.")

targets = pd.read_csv(TARGETS_PATH).drop(columns=["Unnamed: 0"], errors="ignore")
targets["series_id"] = targets["series_id"].map(norm_id)
for c in ["V_EDH","V_SDH","V_IPH","V_SAH","V_IVH","fracture_prob","MLS_mm"]: targets[c] = pd.to_numeric(targets[c], errors="raise")
targets["triage_class"] = pd.to_numeric(targets["triage_class"], errors="raise").astype(int)

ICH = ["V_EDH","V_SDH","V_IPH","V_SAH","V_IVH"]
def add_features(df):
    x=df.copy(); total=x[ICH].sum(axis=1)
    x["feature_any_ich"]=(total>=0.1).astype(int); x["feature_fracture"]=(x["fracture_prob"]>=0.5).astype(int)
    for c in ICH: x[f"feature_{c}"]=(x[c]>=0.1).astype(int)
    x["mls_bin"]=pd.cut(x["MLS_mm"],[-0.01,1,3,5,np.inf],labels=False,include_lowest=True).astype(int)
    for k in [0,1,2]: x[f"feature_triage_{k}"]=(x["triage_class"]==k).astype(int)
    for k in [0,1,2,3]: x[f"feature_mls_bin_{k}"]=(x["mls_bin"]==k).astype(int)
    return x

BAL=["feature_triage_0","feature_triage_1","feature_triage_2","feature_any_ich","feature_fracture","feature_V_EDH","feature_V_SDH","feature_V_IPH","feature_V_SAH","feature_V_IVH","feature_mls_bin_0","feature_mls_bin_1","feature_mls_bin_2","feature_mls_bin_3"]
def choose(df,n,seed0,trials):
    full=df[BAL].mean(); req=["feature_fracture","feature_V_EDH","feature_V_SDH","feature_V_IPH","feature_V_SAH","feature_V_IVH","feature_mls_bin_1","feature_mls_bin_2","feature_mls_bin_3"]; best=None
    for s in range(seed0,seed0+trials):
        rem,sub=train_test_split(df,test_size=n,random_state=s,shuffle=True,stratify=df["triage_class"])
        if any(sub[c].sum()==0 or rem[c].sum()==0 for c in req): continue
        score=float((sub[BAL].mean()-full).abs().mean()+(rem[BAL].mean()-full).abs().mean())
        if best is None or score<best[0]: best=(score,s,rem.copy(),sub.copy())
    if best is None: raise RuntimeError("Split reconstruction failed.")
    return best

sf=add_features(targets)
_,_,train_dev,test_df=choose(sf,N_TEST,SEED,N_SPLIT_TRIALS)
_,_,train_df,dev_df=choose(train_dev.reset_index(drop=True),N_DEV,SEED+N_SPLIT_TRIALS+1,N_SPLIT_TRIALS)
TRAIN_IDS=set(train_df.series_id); DEV_IDS=set(dev_df.series_id)
print("TRAIN",len(TRAIN_IDS),"DEV",len(DEV_IDS))

## 5. Build slice index with fracture labels

In [ ]:
CACHE = CACHE_DIR/"fracture_slice_index.pkl"

def pos_scalar(ds):
    try:
        ori=np.asarray(ds.ImageOrientationPatient,dtype=float); pos=np.asarray(ds.ImagePositionPatient,dtype=float)
        return float(np.dot(pos,np.cross(ori[:3],ori[3:6])))
    except Exception: return np.nan

if CACHE.exists():
    index_df=pd.read_pickle(CACHE)
else:
    rows=[]
    for series_dir in tqdm(sorted([p for p in TRAINING_DIR.iterdir() if p.is_dir()],key=lambda p:int(p.name)),desc="Indexing"):
        sid=norm_id(series_dir.name)
        if sid not in TRAIN_IDS and sid not in DEV_IDS: continue
        split="TRAIN" if sid in TRAIN_IDS else "DEV"
        true_frac=int(float(targets.loc[targets.series_id==sid,"fracture_prob"].iloc[0])>=0.5)
        for dcm in series_dir.glob("*.dcm"):
            ds=pydicom.dcmread(dcm,stop_before_pixels=True,force=True); uid=str(getattr(ds,"SOPInstanceUID",dcm.stem))
            ann_path=ANNOTATIONS_DIR/sid/f"{uid}.json"; ann_exists=int(ann_path.exists()); n_boxes=np.nan
            if ann_path.exists():
                with open(ann_path,"r",encoding="utf-8") as f: ann=json.load(f)
                n_boxes=len(ann.get("boxes_xywh",[]) or [])
            rows.append({"series_id":sid,"split":split,"dicom_path":str(dcm),"sop_uid":uid,"position":pos_scalar(ds),"instance":float(getattr(ds,"InstanceNumber",np.nan)),"true_fracture":true_frac,"annotation_exists":ann_exists,"n_boxes":n_boxes})
    index_df=pd.DataFrame(rows); index_df.to_pickle(CACHE)

groups={}; lookup={}
for sid,g in index_df.groupby("series_id"):
    g=g.copy()
    if g.position.notna().all(): g=g.sort_values("position")
    elif g.instance.notna().all(): g=g.sort_values("instance")
    else: g=g.sort_values("dicom_path")
    g=g.reset_index(drop=True); groups[sid]=g
    for i,r in g.iterrows(): lookup[(sid,str(r.sop_uid))]=i

train_pos=index_df[(index_df["split"]=="TRAIN")&(index_df["true_fracture"]==1)&(index_df["annotation_exists"]==1)].copy()
train_neg=index_df[(index_df["split"]=="TRAIN")&(index_df["true_fracture"]==0)].copy()
train_slices=pd.concat([train_pos,train_neg],ignore_index=True)
train_slices["label"]=np.where(train_slices["true_fracture"]==0,0,(train_slices["n_boxes"].fillna(0)>0).astype(int))
dev_slices=index_df[index_df["split"]=="DEV"].copy()
print(train_slices["label"].value_counts())

## 6. Dataset and model

In [ ]:
@lru_cache(maxsize=HU_CACHE_SIZE)
def load_hu(path):
    ds=pydicom.dcmread(path,force=True); return np.asarray(apply_modality_lut(ds.pixel_array,ds),dtype=np.float32)

def bone(x):
    lo=BONE_CENTER-BONE_WIDTH/2; hi=BONE_CENTER+BONE_WIDTH/2
    t=torch.from_numpy(((np.clip(x,lo,hi)-lo)/(hi-lo)).astype(np.float32))
    return F.interpolate(t[None,None],size=(IMAGE_SIZE,IMAGE_SIZE),mode="bilinear",align_corners=False)[0,0]

class FractureDataset(Dataset):
    def __init__(self,df,train=False): self.df=df.reset_index(drop=True); self.train=train
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; g=groups[r.series_id]; ci=lookup[(r.series_id,str(r.sop_uid))]
        idx=[min(max(ci+o,0),len(g)-1) for o in CONTEXT_OFFSETS]
        x=torch.stack([bone(load_hu(str(g.iloc[j].dicom_path))) for j in idx])
        if self.train and random.random()<0.5: x=torch.flip(x,[-1])
        x=(x-0.5)/0.25
        return x, torch.tensor(float(r.label)), r.series_id

series_counts=train_slices.groupby("series_id").size().to_dict()
class_counts=train_slices["label"].value_counts().to_dict()
weights=[(1/series_counts[r.series_id])*(1/class_counts[int(r.label)]) for r in train_slices.itertuples()]
sampler=WeightedRandomSampler(torch.tensor(weights,dtype=torch.double),SAMPLES_PER_EPOCH,replacement=True)
train_loader=DataLoader(FractureDataset(train_slices,True),batch_size=BATCH_SIZE,sampler=sampler,num_workers=NUM_WORKERS,pin_memory=True)

model=efficientnet_b2(weights=EfficientNet_B2_Weights.DEFAULT)
model.classifier[1]=nn.Linear(model.classifier[1].in_features,1)
model=model.to(DEVICE)

## 7. Train

In [ ]:
opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
scaler=torch.amp.GradScaler("cuda",enabled=DEVICE.type=="cuda")
best_f1=-1; stale=0; best_path=MODELS_DIR/"fracture_2p5d_efficientnet_b2_best.pth"

def series_scores():
    model.eval(); rows=[]
    with torch.inference_mode():
        for sid in tqdm(sorted(DEV_IDS),desc="DEV inference",leave=False):
            g=groups[sid]; scores=[]
            for start in range(0,len(g),BATCH_SIZE):
                batch=[]
                for ci in range(start,min(start+BATCH_SIZE,len(g))):
                    idx=[min(max(ci+o,0),len(g)-1) for o in CONTEXT_OFFSETS]
                    x=torch.stack([bone(load_hu(str(g.iloc[j].dicom_path))) for j in idx]); batch.append((x-0.5)/0.25)
                p=torch.sigmoid(model(torch.stack(batch).to(DEVICE)).squeeze(1)).cpu().numpy(); scores.extend(p.tolist())
            scores=np.asarray(scores)
            row={"series_id":sid,"true":int(float(targets.loc[targets.series_id==sid,"fracture_prob"].iloc[0])>=0.5),"max":float(scores.max())}
            for k in [3,5,10]:
                kk=min(k,len(scores)); row[f"top{k}_mean"]=float(np.sort(scores)[-kk:].mean())
            rows.append(row)
    return pd.DataFrame(rows)

for epoch in range(EPOCHS):
    model.train(); losses=[]
    for x,y,_ in tqdm(train_loader,desc=f"epoch {epoch+1}/{EPOCHS}",leave=False):
        x=x.to(DEVICE); y=y.to(DEVICE); opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda",enabled=DEVICE.type=="cuda"):
            logits=model(x).squeeze(1); loss=F.binary_cross_entropy_with_logits(logits,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); losses.append(loss.item())
    dev_scores=series_scores(); best_epoch_f1=0
    for col in ["max","top3_mean","top5_mean","top10_mean"]:
        for th in np.arange(0.05,0.96,0.05):
            pred=(dev_scores[col]>=th).astype(int); best_epoch_f1=max(best_epoch_f1,f1_score(dev_scores.true,pred,zero_division=0))
    print(f"Epoch {epoch+1:02d} loss={np.mean(losses):.4f} DEV best series F1={best_epoch_f1:.4f}")
    if best_epoch_f1>best_f1:
        best_f1=best_epoch_f1; stale=0; torch.save(model.state_dict(),best_path)
    else: stale+=1
    if stale>=PATIENCE: break

model.load_state_dict(torch.load(best_path,map_location=DEVICE)); model.eval()
print("Best DEV series F1 during training:",best_f1)

## 8. Final DEV aggregation search

In [ ]:
dev_scores=series_scores(); rows=[]
for col in ["max","top3_mean","top5_mean","top10_mean"]:
    for th in np.arange(0.01,1.0,0.01):
        pred=(dev_scores[col]>=th).astype(int); tn,fp,fn,tp=confusion_matrix(dev_scores.true,pred,labels=[0,1]).ravel()
        rows.append({"aggregator":col,"threshold":th,"F1":f1_score(dev_scores.true,pred,zero_division=0),"precision":precision_score(dev_scores.true,pred,zero_division=0),"recall":recall_score(dev_scores.true,pred,zero_division=0),"FP":fp,"FN":fn})
result=pd.DataFrame(rows).sort_values(["F1","recall","precision"],ascending=False).reset_index(drop=True)
result.to_csv(METRICS_DIR/"aggregation_search.csv",index=False); dev_scores.to_csv(METRICS_DIR/"dev_slice_aggregates.csv",index=False)
display(result.head(20))

## 9. Save model and config

In [ ]:
best=result.iloc[0]
torch.save(model.state_dict(),MODELS_DIR/"fracture_2p5d_efficientnet_b2_final.pth")
config={"architecture":"efficientnet_b2","input":"previous_center_next_bone","image_size":IMAGE_SIZE,"bone_window":[BONE_CENTER,BONE_WIDTH],"aggregator":best["aggregator"],"threshold":float(best["threshold"]),"DEV_F1":float(best["F1"])}
with open(MODELS_DIR/"fracture_2p5d_config.json","w") as f: json.dump(config,f,indent=2)
pd.DataFrame([config]).to_csv(OUTPUT_ROOT/"00_DIRECT_ANSWERS.csv",index=False)
print(config)